# Baselines: RANDOM y GREEDY
Comparar las implementaciones integradas sin repetir búsquedas costosas. Ambos métodos usan el mismo simulador, K=5 y el mismo calendario de 18 días. Los resultados archivados pertenecen al motor 1.1.0; el motor acelerado 1.1.1 se evalúa por separado.

RANDOM selecciona K identificadores válidos sin reemplazo; su semilla de selección es distinta de las seeds del simulador. GREEDY agrega el sitio con mayor ganancia de accesibilidad floral ponderada por distancia y cobertura ya seleccionada. Su criterio es una heurística espacial, no una predicción de néctar observado.

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'config/project2/base.yaml').exists())
sys.path.insert(0, str(ROOT / 'src'))
from bee_sim.analysis.project2_comparison import build_comparison
data = build_comparison(ROOT)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})


In [2]:
display(data['audit'].query("method != 'NSGA2'"))
display(data['summary'].query("method != 'NSGA2'")[["method","metric","mean","ci95_low","ci95_high","n_seeds"]])

,method,file,original_rows,holdout_rows,solutions,holdout_replications,sha256,config_hash,engine_hash,input_hash,metric_contract_hash,engine_version,K,scenario,mode,resource_unit
0,RANDOM,random_results.csv,3000,2100,30,70,1ea1b9c7e5dc5bf52b604d6b02c4fb4cff7da32f687793...,32b62484aff30d3403707694d538d776600f9dec8381d4...,6a032bf015293cabcaa47dc8b0a3cd72e8b8d3e1061dee...,df45f2a5654d783531ea18cb0fb116eab76f3c215b8499...,290ae72c96171b1ab6055b939488527f02d4eab04cfd52...,project2-synthetic-1.1.0,5,synthetic_base_v1,synthetic,synthetic_RU
1,GREEDY,greedy_results.csv,100,70,1,70,de94eeac4dd5ee43a26246dc57908c0211351fdeec1547...,32b62484aff30d3403707694d538d776600f9dec8381d4...,6a032bf015293cabcaa47dc8b0a3cd72e8b8d3e1061dee...,df45f2a5654d783531ea18cb0fb116eab76f3c215b8499...,290ae72c96171b1ab6055b939488527f02d4eab04cfd52...,project2-synthetic-1.1.0,5,synthetic_base_v1,synthetic,synthetic_RU


,method,metric,mean,ci95_low,ci95_high,n_seeds
0,RANDOM,total_resource_obtained,7.216620e+02,7.176912e+02,7.257656e+02,70
1,RANDOM,mean_resource_per_hive,1.443324e+02,1.435382e+02,1.451531e+02,70
2,RANDOM,min_resource_per_hive,1.061084e+02,1.056056e+02,1.066170e+02,70
3,RANDOM,mean_distance_m,1.618409e+03,1.617146e+03,1.619605e+03,70
4,RANDOM,total_distance_m,1.754259e+06,1.741466e+06,1.767471e+06,70
5,RANDOM,p95_distance_m,2.852905e+03,2.851711e+03,2.854029e+03,70
6,RANDOM,competition_pressure,3.218465e-04,2.926669e-04,3.540396e-04,70
7,RANDOM,hive_balance_jain,9.549280e-01,9.541059e-01,9.557910e-01,70
8,RANDOM,patches_used_count,2.393571e+02,2.385585e+02,2.401933e+02,70
9,GREEDY,total_resource_obtained,9.143756e+02,9.068172e+02,9.221132e+02,70


## Verificar instalaciones
Se conservan las 30 instalaciones RANDOM y la instalación GREEDY de los autores. Identificadores distintos no implican necesariamente coordenadas distintas: esta política debe decidirse explícitamente para futuros experimentos.

In [3]:
candidates = pd.read_csv(ROOT / 'data/project2/processed/candidate_hive_sites.csv')
for method in ['RANDOM', 'GREEDY']:
    for _, group in data['frames'][method].groupby('solution_id'):
        sites = json.loads(group.site_ids.iloc[0])
        assert len(sites) == len(set(sites)) == 5
        assert set(sites) <= set(candidates.site_id)
print('Instalaciones válidas; 70 seeds comunes por instalación.')
display(candidates[['x_m','y_m']].duplicated().value_counts().rename_axis('coordenada_repetida').to_frame('candidatos'))

Instalaciones válidas; 70 seeds comunes por instalación.


,candidatos
coordenada_repetida,
False,351
True,24


## Interpretación
RANDOM se promedia primero entre sus instalaciones para cada seed; no se tratan 2 100 filas como observaciones independientes. Los intervalos son condicionales a estas instalaciones guardadas. Falta estudiar variabilidad entre selecciones RANDOM nuevas y calibrar el comportamiento con evidencia ecológica.